In [ ]:
# Block 1
## IMPORT LIBRARIES
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np

In [10]:
## Block 2 -- Globals: normalized scema, header aliases, merged patterns

# Final normalized columns
NORMALIZED_COLS = [
    "season","team","number","name","position","height","weight","class",
    "hometown","high_school","previous_school"
]

# Header synonyms (lowercased; we canonicalize later)
HEADER_ALIASES = {
    "number": {"#", "no", "num", "number", "jersey"},
    "name": {"name", "student-athlete", "student athlete", "player", "full name"},
    "position": {"pos", "position"},
    "height": {"ht", "height", "ht."},
    "weight": {"wt", "weight", "wt."},
    "class": {"class", "yr", "year", "academic year", "eligibility", "elig yr", "cl.", "cl"},
    "hometown": {"hometown", "home town", "city/state"},
    "high_school": {"hs", "highschool", "high school", "hs name"},
    "previous_school": {
        "previous school","prev school","previous","last school","previous college",
        "prev college","college","university","prior school"
    },
}

# Common merged header phrasings
MERGED_PATTERNS = {
    "hometown_high_school": [
        "hometown/high school","hometown / high school","home/hs","hometown/hs",
        "city/state/high school","hometown – high school","hometown- high school"
    ],
    "hometown_previous_school": [
        "hometown/previous school","hometown / previous school","home/previous school",
        "hometown/last school","hometown / last school","city/state/previous school",
        "hometown/previous college","hometown / previous college"
    ],
}


In [11]:
#Block 3 -- Small Text Helpers

def _canon_text(s: str) -> str:
    if s is None: return ""
    return re.sub(r"\s+", " ", s).strip()

def _canon_key(s: str) -> str:
    return re.sub(r"[^\w]+", "", (s or "").lower())

def _split_pair(val: str):
    """Split 'City, ST / RightSide' into (left, right)."""
    if not val: return (None, None)
    s = _canon_text(val)
    parts = [p.strip(" /–—|-") for p in re.split(r"\s*[\/–—|-]\s*", s) if p.strip(" /–—|-")]
    if len(parts) >= 2:
        return (_canon_text(parts[0]), _canon_text(" - ".join(parts[1:])))
    return (_canon_text(s), None)

def _to_int(x):
    if x is None: return None
    m = re.search(r"\d+", str(x))
    return int(m.group()) if m else None


In [12]:
#Block 4 -- Detect table + map headers to normalized keys

def _header_to_key(h: str) -> str | None:
    """Map a header cell's text to a normalized field or merged token."""
    h_clean = _canon_text(h)
    hk = h_clean.lower()

    # merged forms first
    for token, patterns in MERGED_PATTERNS.items():
        for p in patterns:
            if p in hk:
                return token
    # heuristic merged detection
    if "hometown" in hk and "high" in hk and "school" in hk:
        return "hometown_high_school"
    if "hometown" in hk and any(w in hk for w in ["previous","last","college","university","prev"]):
        return "hometown_previous_school"

    # direct alias lookup
    ck = _canon_key(h_clean)
    for norm, aliases in HEADER_ALIASES.items():
        if ck in {_canon_key(a) for a in aliases}:
            return norm
    return None

def _find_roster_table(soup: BeautifulSoup):
    """Pick the table that *looks* like the roster by scanning headers."""
    candidate_tables = soup.find_all("table")
    best = None; best_score = -1
    keywords = {"name","position","hometown","high","previous","class","#","no","height","weight"}

    for tbl in candidate_tables:
        ths = [th.get_text(" ", strip=True) for th in tbl.find_all("th")]
        if not ths:
            first_tr = tbl.find("tr")
            if first_tr:
                ths = [td.get_text(" ", strip=True) for td in first_tr.find_all("td")]
        if not ths:
            continue
        head_text = " ".join(ths).lower()
        score = sum(1 for k in keywords if k in head_text)
        if score > best_score:
            best_score = score
            best = (tbl, ths)
    return best  # (table, headers) or None

def _build_keymap(headers: list[str]) -> list[str | None]:
    """Return list aligned with headers; values are normalized keys or merged tokens."""
    return [_header_to_key(h) for h in headers]


In [13]:
## Block 5 -- Core parser: grid-only, header-driven, transfer-aware

def parse_swac_grid(url: str, season: int, team: str) -> pd.DataFrame:
    html = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")

    found = _find_roster_table(soup)
    if not found:
        return pd.DataFrame(columns=NORMALIZED_COLS)

    table, headers = found
    keymap = _build_keymap(headers)

    # Choose rows (tbody preferred)
    body = table.find("tbody") or table
    trs = body.find_all("tr")

    # If first row looks like headers in <td>, skip it
    if trs and headers and all(el.name == "td" for el in (table.find("tr") or body).find_all(["th","td"])):
        trs = trs[1:]

    rows = []
    for tr in trs:
        cells = tr.find_all(["td","th"])
        if not cells:
            continue
        raw = {}
        for i, td in enumerate(cells[:len(keymap)]):
            k = keymap[i]
            if not k: 
                continue
            text = td.get_text(" ", strip=True)
            if k in {"hometown_high_school","hometown_previous_school"}:
                raw[k] = text
            else:
                if k == "name":
                    a = td.find("a")
                    text = a.get_text(" ", strip=True) if a else text
                raw[k] = text

        # Resolve merged vs separate
        hometown = raw.get("hometown")
        high_school = raw.get("high_school")
        previous_school = raw.get("previous_school")

        if raw.get("hometown_high_school"):
            h, hs = _split_pair(raw["hometown_high_school"])
            hometown = hometown or h
            high_school = high_school or hs

        if raw.get("hometown_previous_school"):
            h, prev = _split_pair(raw["hometown_previous_school"])
            hometown = hometown or h
            previous_school = previous_school or prev

        rows.append({
            "season": season,
            "team": team,
            "number": _to_int(raw.get("number")),
            "name": _canon_text(raw.get("name")),
            "position": _canon_text(raw.get("position")),
            "height": _canon_text(raw.get("height")),
            "weight": _canon_text(raw.get("weight")),
            "class": _canon_text(raw.get("class")),
            "hometown": _canon_text(hometown),
            "high_school": _canon_text(high_school),
            "previous_school": _canon_text(previous_school),
        })

    df = pd.DataFrame(rows, columns=NORMALIZED_COLS).replace({"": pd.NA})
    if not df.empty:
        df = df.drop_duplicates(subset=["team","season","name","number"], keep="first")
    return df


In [14]:
## Block 6 -- Quick Header Preview + sanity checks 

def preview_headers(url: str):
    html = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    found = _find_roster_table(soup)
    if not found:
        print("No roster-like table found.")
        return
    _, headers = found
    keymap = _build_keymap(headers)
    print("HEADERS:", headers)
    print("KEYMAP :", keymap)

def peek_df(df: pd.DataFrame, n=5):
    if df.empty:
        print("⚠️ DataFrame is EMPTY")
        return
    print(df.head(n))
    print("\nNull % by column:")
    print((df.isna().mean()*100).round(1).astype(str) + "%")


In [15]:
## Block 7 -- Year Loop with smart slugs and URL fallbacks

from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

# ---- Your existing knobs (paste from your message or keep as-is) ----
SCHOOLS = {
    "Jackson State": "https://gojsutigers.com/sports/football/roster/{year}?view=2",
    "Alabama State": "https://bamastatesports.com/sports/football/roster/{year}?view=2",
    "Alabama A&M": "https://aamusports.com/sports/football/roster/{year}?view=2",
    "Southern": "https://gojagsports.com/sports/football/roster/{year}?view=2",
    "Prairie View A&M": "https://pvpanthers.com/sports/football/roster/{year}?view=2",
    "Texas Southern": "https://tsusports.com/sports/football/roster/{year}?view=2",
    "UAPB": "https://uapblionsroar.com/sports/football/roster/{year}?view=2",
    "Alcorn State": "https://alcornsports.com/sports/football/roster/{year}?view=2",
    "Grambling": "https://gsutigers.com/sports/football/roster/{year}?view=2",
    "Mississippi Valley State": "https://mvsusports.com/sports/football/roster/{year}?view=2",
    "Florida A&M": "https://famuathletics.com/sports/football/roster/{year}?view=2",
    "Bethune-Cookman": "https://bcuathletics.com/sports/football/roster/{year}?view=2"
}
YEARS = list(range(2010, 2026))
HEADLESS = True
PAGE_LOAD_TIMEOUT = 20
POLITE_DELAY = 1.5
EMPTY_ROW_THRESHOLD = 1
AUTO_STOP_CONSECUTIVE = 4

# ---- Helpers for slug & URL variants ----
def build_year_slugs(y: int):
    """
    Return a small list of plausible slugs for a football season 'y'.
    Covers:
      - '2022'
      - '2021-22'  (academic year spanning y-1 to y)
      - '2022-23'  (some sites tag the roster this way)
    """
    yy = f"{y}"
    left = f"{y-1}-{str(y % 100).zfill(2)}"
    right = f"{y}-{str((y+1) % 100).zfill(2)}"
    # unique, preserve order
    seen = set()
    out = [s for s in [yy, left, right] if not (s in seen or seen.add(s))]
    return out

def url_variants_from_template(template: str, slug: str):
    """
    Given 'https://.../roster/{year}?view=2', produce variants like:
     - with '{year}' replaced by slug
     - also a no-params version
     - also if the template has no '?view=2', add one variant with it
    """
    base = template.format(year=slug)
    urls = [base]

    # parse and toggle view=2
    parsed = urlparse(base)
    q = dict(parse_qsl(parsed.query))
    if "view" in q:
        # variant with view removed
        if q:
            q2 = dict(q); q2.pop("view", None)
            url_no_view = urlunparse(parsed._replace(query=urlencode(q2)))
            urls.append(url_no_view)
    else:
        # variant with view=2 added
        q2 = dict(q); q2["view"] = "2"
        url_with_view = urlunparse(parsed._replace(query=urlencode(q2)))
        urls.append(url_with_view)

    # unique, preserve order
    uniq = []
    seen = set()
    for u in urls:
        if u not in seen:
            uniq.append(u); seen.add(u)
    return uniq

def try_parse_any(urls, season, team):
    """
    Try multiple URL candidates; return (df, used_url) for the first non-empty parse.
    If all parse empty, return (empty_df, last_url_tried).
    """
    last_url = None
    for u in urls:
        last_url = u
        df = parse_swac_grid(u, season=season, team=team)
        if not df.empty and len(df) > EMPTY_ROW_THRESHOLD:
            df["source_url"] = u
            return df, u
    # return the "best we got" (might be empty)
    df = parse_swac_grid(last_url, season=season, team=team) if last_url else pd.DataFrame(columns=NORMALIZED_COLS)
    if not df.empty:
        df["source_url"] = last_url
    return df, last_url

def log_preview(df: pd.DataFrame, label: str, n=5):
    print(f"\nPreview — {label}")
    if df.empty:
        print("⚠️ EMPTY")
        return
    print(df.head(n))
    nulls = (df.isna().mean()*100).round(1).astype(str) + "%"
    print("\nNull % by column:\n", nulls[["hometown","high_school","previous_school"]])


In [ ]:
### Block 8 Run Loop

from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
import os
import re
import time

# === Output directory (same as you asked) ===
OUTPUT_DIR = r"C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC Rosters"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Your knobs (can keep as-is) ----
HEADLESS = True
PAGE_LOAD_TIMEOUT = 20
POLITE_DELAY = 1.0
EMPTY_ROW_THRESHOLD = 1
AUTO_STOP_CONSECUTIVE = 4

# YEARS newest -> oldest
YEARS = list(range(2010, 2026))
YEARS = list(sorted(YEARS, reverse=True))  # NEW: iterate from most recent down

def build_year_slugs(y: int):
    yy = f"{y}"
    left = f"{y-1}-{str(y % 100).zfill(2)}"
    right = f"{y}-{str((y+1) % 100).zfill(2)}"
    seen = set()
    out = [s for s in [yy, left, right] if not (s in seen or seen.add(s))]
    return out

def url_variants_from_template(template: str, slug: str):
    base = template.format(year=slug)
    urls = [base]
    parsed = urlparse(base)
    q = dict(parse_qsl(parsed.query))
    if "view" in q:
        q2 = dict(q); q2.pop("view", None)
        url_no_view = urlunparse(parsed._replace(query=urlencode(q2)))
        urls.append(url_no_view)
    else:
        q2 = dict(q); q2["view"] = "2"
        url_with_view = urlunparse(parsed._replace(query=urlencode(q2)))
        urls.append(url_with_view)
    uniq, seen = [], set()
    for u in urls:
        if u not in seen:
            uniq.append(u); seen.add(u)
    return uniq

def try_parse_any(urls, season, team):
    last_url = None
    for u in urls:
        last_url = u
        df = parse_swac_grid(u, season=season, team=team)
        if not df.empty and len(df) > EMPTY_ROW_THRESHOLD:
            df["source_url"] = u
            return df, u
    df = parse_swac_grid(last_url, season=season, team=team) if last_url else pd.DataFrame(columns=NORMALIZED_COLS)
    if not df.empty:
        df["source_url"] = last_url
    return df, last_url

def log_preview(df: pd.DataFrame, label: str, n=5):
    print(f"\nPreview — {label}")
    if df.empty:
        print("⚠️ EMPTY")
        return
    print(df.head(n))
    nulls = (df.isna().mean()*100).round(1).astype(str) + "%"
    print("\nNull % by column:\n", nulls[["hometown","high_school","previous_school"]])

# === Run the loop ===
all_chunks = []
for school, template in SCHOOLS.items():
    print(f"\n===== {school} =====")
    school_chunks = []       # collect this school's seasons
    empty_streak = 0

    for y in YEARS:
        slugs = build_year_slugs(y)
        candidate_urls = []
        for s in slugs:
            candidate_urls.extend(url_variants_from_template(template, s))
        # unique preserve order
        seen = set(); candidate_urls = [u for u in candidate_urls if not (u in seen or seen.add(u))]

        print(f"\n{school} — Season {y}")
        print("Trying:", " | ".join(candidate_urls[:3]) + (" ..." if len(candidate_urls) > 3 else ""))

        try:
            df, used = try_parse_any(candidate_urls, season=y, team=school)
            label = f"{school} {y} ({'OK' if not df.empty else 'EMPTY'})"
            log_preview(df, label, n=5)

            if df.empty or len(df) <= EMPTY_ROW_THRESHOLD:
                empty_streak += 1
                print(f"⚠️ No usable rows for {school} {y}. Empty streak = {empty_streak}")
            else:
                empty_streak = 0
                # sanity warning
                if df["high_school"].isna().all() and df["previous_school"].isna().all():
                    print("⚠️ All HS & Previous School are null — check headers/merged patterns:", used)
                school_chunks.append(df)

            time.sleep(POLITE_DELAY)

            if AUTO_STOP_CONSECUTIVE and empty_streak >= AUTO_STOP_CONSECUTIVE:
                print(f"⛳ Auto-stop: {school} hit {AUTO_STOP_CONSECUTIVE} empty years in a row. Moving on.")
                break

        except Exception as e:
            empty_streak += 1
            print(f"❌ ERROR {school} {y}: {e}")
            if AUTO_STOP_CONSECUTIVE and empty_streak >= AUTO_STOP_CONSECUTIVE:
                print(f"⛳ Auto-stop after errors/empties: {school}")
                break

    # === Save this school's results immediately (NEW) ===
    if school_chunks:
        school_df = pd.concat(school_chunks, ignore_index=True)
        # save combined-by-school
        safe_team = re.sub(r"[^A-Za-z0-9]+", "_", school)
        combined_path = os.path.join(OUTPUT_DIR, f"{safe_team}_ALL_Seasons_roster.csv")
        school_df.to_csv(combined_path, index=False, encoding="utf-8-sig")
        print(f"💾 Saved (school combined): {combined_path} ({school_df.shape[0]} rows)")

        # also save per-season files (newest→oldest order)
        for season in sorted(school_df["season"].dropna().unique(), reverse=True):
            subset = school_df[school_df["season"] == season]
            file_name = f"{safe_team}_{season}_roster.csv"
            subset.to_csv(os.path.join(OUTPUT_DIR, file_name), index=False, encoding="utf-8-sig")
            print(f"  └─ Saved: {file_name} ({subset.shape[0]} rows)")

        # append to global master
        all_chunks.append(school_df)
    else:
        print(f"ℹ️ No data saved for {school}.")

# Build global combined (can be empty)
combined = pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame(columns=NORMALIZED_COLS + ["source_url"])
print("\n=== Combined shape:", combined.shape, "===")
log_preview(combined, "COMBINED", n=10)

## Took 7 minutes 19 seconds to run


===== Jackson State =====

Jackson State — Season 2025
Trying: https://gojsutigers.com/sports/football/roster/2025?view=2 | https://gojsutigers.com/sports/football/roster/2025 | https://gojsutigers.com/sports/football/roster/2024-25?view=2 ...

Preview — Jackson State 2025 (OK)
   season           team  number                name position height weight  \
0    2025  Jackson State       0  Travis Terrell Jr.       RB    5-9    170   
1    2025  Jackson State       0   Jeremiah Williams       DL    6-2    314   
2    2025  Jackson State       1     Khamauri Rogers       DB    6-0    173   
3    2025  Jackson State       1       Shemar Savage       WR    6-3    216   
4    2025  Jackson State       2    Ja'Naylon Dupree       WR    5-8    162   

   class             hometown               high_school  \
0    So.         Atlanta, Ga.              Creekside HS   
1  R-Sr.     Lexington, Miss.  Holmes County Central HS   
2    Gr.       Madison, Miss.  Holmes County Central HS   
3    Gr. 

C:\Users\jland\AppData\Local\Temp\ipykernel_27900\4179749789.py:120: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  school_df = pd.concat(school_chunks, ignore_index=True)


💾 Saved (school combined): C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\Texas_Southern_ALL_Seasons_roster.csv (1385 rows)
  └─ Saved: Texas_Southern_2025_roster.csv (98 rows)
  └─ Saved: Texas_Southern_2024_roster.csv (102 rows)
  └─ Saved: Texas_Southern_2023_roster.csv (97 rows)
  └─ Saved: Texas_Southern_2022_roster.csv (93 rows)
  └─ Saved: Texas_Southern_2021_roster.csv (104 rows)
  └─ Saved: Texas_Southern_2020_roster.csv (11 rows)
  └─ Saved: Texas_Southern_2019_roster.csv (93 rows)
  └─ Saved: Texas_Southern_2018_roster.csv (90 rows)
  └─ Saved: Texas_Southern_2017_roster.csv (90 rows)
  └─ Saved: Texas_Southern_2016_roster.csv (84 rows)
  └─ Saved: Texas_Southern_2015_roster.csv (70 rows)
  └─ Saved: Texas_Southern_2014_roster.csv (80 rows)
  └─ Saved: Texas_Southern_2013_roster.csv (79 rows)
  └─ Saved: Texas_Southern_2012_roster.csv (98 rows)
  └─ Saved: Texas_Southern_2011_roster.csv (98 rows)
  └─ Saved: Texas_Southern_2010_roster.csv (98 rows)

In [17]:
### Block 9 -- Save to CSV
# Save one master file after all schools are processed
if not combined.empty:
    master_path = os.path.join(OUTPUT_DIR, "SWAC_Rosters_Combined.csv")
    combined.to_csv(master_path, index=False, encoding="utf-8-sig")
    print(f"\n📦 Master file saved: {master_path} ({combined.shape[0]} rows)")
else:
    print("⚠️ No combined data to save.")




📦 Master file saved: C:\Users\jland\OneDrive\Emory Essentials\Research\College Football Output\SWAC_Rosters_Combined.csv (17685 rows)
